# Importing the Libraries and Datasets

In [2]:
import os, warnings, json as _json
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.model_selection import train_test_split, cross_val_score, RandomizedSearchCV
from sklearn.inspection import permutation_importance
from sklearn.linear_model import Ridge
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization, LeakyReLU
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.regularizers import l2
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, f1_score
from scipy.stats import randint as sp_randint
from scipy.optimize import minimize

import joblib

RANDOM_STATE = 42

In [16]:
df1_raw= pd.read_csv('data/1. Gym Members Exercise Dataset/gym_members_exercise_tracking.csv')
df2_raw = pd.read_csv('data/2. Exercise and Fitness Metrics Dataset/exercise_dataset.csv')

print("Columns in gym_members_exercise_tracking.csv:", df_gym.columns.tolist())
print("Columns in exercise_dataset.csv:", df_fit.columns.tolist())

print()

print(f'Dataset 1 (Gym Members) Shape : {df_gym.shape[0]:,} rows × {df_gym.shape[1]} cols')
print(f'Dataset 2 (Exercise & Fitness) Shape : {df_fit.shape[0]:,} rows × {df_fit.shape[1]} cols')
print()

Columns in gym_members_exercise_tracking.csv: ['Age', 'Gender', 'Weight (kg)', 'Height (m)', 'Max_BPM', 'Avg_BPM', 'Resting_BPM', 'Session_Duration (hours)', 'Calories_Burned', 'Workout_Type', 'Fat_Percentage', 'Water_Intake (liters)', 'Workout_Frequency (days/week)', 'Experience_Level', 'BMI']
Columns in exercise_dataset.csv: ['ID', 'Exercise', 'Calories Burn', 'Dream Weight', 'Actual Weight', 'Age', 'Gender', 'Duration', 'Heart Rate', 'BMI', 'Weather Conditions', 'Exercise Intensity']

Dataset 1 (Gym Members) Shape : 973 rows × 15 cols
Dataset 2 (Exercise & Fitness) Shape : 3,864 rows × 12 cols



In [17]:
print('Gym Members Exercise Tracking Dataset (1)')
display(df1_raw.head(3))
print('Exercise and Fitness Metrics Dataset (2)')
display(df2_raw.head(3))

Gym Members Exercise Tracking Dataset (1)


,Age,Gender,Weight (kg),Height (m),Max_BPM,Avg_BPM,Resting_BPM,Session_Duration (hours),Calories_Burned,Workout_Type,Fat_Percentage,Water_Intake (liters),Workout_Frequency (days/week),Experience_Level,BMI
0,56,Male,88.3,1.71,180,157,60,1.69,1313.0,Yoga,12.6,3.5,4,3,30.20
1,46,Female,74.9,1.53,179,151,66,1.30,883.0,HIIT,33.9,2.1,4,2,32.00
2,32,Female,68.1,1.66,167,122,54,1.11,677.0,Cardio,33.4,2.3,4,2,24.71


Exercise and Fitness Metrics Dataset (2)


,ID,Exercise,Calories Burn,Dream Weight,Actual Weight,Age,Gender,Duration,Heart Rate,BMI,Weather Conditions,Exercise Intensity
0,1,Exercise 2,286.959851,91.892531,96.301115,45,Male,37,170,29.426275,Rainy,5
1,2,Exercise 7,343.453036,64.165097,61.104668,25,Male,43,142,21.286346,Rainy,5
2,3,Exercise 4,261.223465,70.846224,71.766724,20,Male,20,148,27.899592,Cloudy,4


In [20]:
print('Dataset 1: Calories_Burned stats:')
print(df1_raw['Calories_Burned'].describe().round(1))
print()
print('Dataset 2: Calories Burn stats:')
print(df2_raw['Calories Burn'].describe().round(1))
print()

Dataset 1: Calories_Burned stats:
count     973.0
mean      905.4
std       272.6
min       303.0
25%       720.0
50%       893.0
75%      1076.0
max      1783.0
Name: Calories_Burned, dtype: float64

Dataset 2: Calories Burn stats:
count    3864.0
mean      301.9
std       115.8
min       100.0
25%       202.2
50%       299.7
75%       404.1
max       499.9
Name: Calories Burn, dtype: float64



In [53]:
print('Dataset 1 info:')
print(df1_raw.info())
print()
print('Dataset 2 info:')
print(df2_raw.info())
print()

Dataset 1 info:
<class 'pandas.DataFrame'>
RangeIndex: 973 entries, 0 to 972
Data columns (total 15 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   Age                            973 non-null    int64  
 1   Gender                         973 non-null    str    
 2   Weight (kg)                    973 non-null    float64
 3   Height (m)                     973 non-null    float64
 4   Max_BPM                        973 non-null    int64  
 5   Avg_BPM                        973 non-null    int64  
 6   Resting_BPM                    973 non-null    int64  
 7   Session_Duration (hours)       973 non-null    float64
 8   Calories_Burned                973 non-null    float64
 9   Workout_Type                   973 non-null    str    
 10  Fat_Percentage                 973 non-null    float64
 11  Water_Intake (liters)          973 non-null    float64
 12  Workout_Frequency (days/week)  973 non-null  

# Cleaning Datasets Separately

In [46]:
df1 = df1_raw.copy()

if df1.isnull().sum().sum()  == 0:
    print("No Null Values")
else:
    print('Null Values Available')

print(f'  Columns in Dataset 1: {df1.columns.tolist()}')

No Null Values
  Columns in Dataset 1: ['Age', 'Gender', 'Weight (kg)', 'Height (m)', 'Max_BPM', 'Avg_BPM', 'Resting_BPM', 'Session_Duration (hours)', 'Calories_Burned', 'Workout_Type', 'Fat_Percentage', 'Water_Intake (liters)', 'Workout_Frequency (days/week)', 'Experience_Level', 'BMI']


In [47]:
df2 = df2_raw.copy()

df2.drop(columns=['ID', 'Exercise', 'Dream Weight'], inplace=True)

#Renaming columns to match in Dataset 1
df2.rename(columns={
    'Calories Burn':'Calories_Burned','Actual Weight':'Weight (kg)','Heart Rate':'Avg_BPM','Weather Conditions':'Weather_Conditions','Exercise Intensity':'Exercise_Intensity','Duration': 'Session_Duration (hours)'
}, inplace=True)

#Dataset 2 duration converting it to hours to match Dataset 1
df2['Session_Duration (hours)'] = df2['Session_Duration (hours)'] / 60

if df2.isnull().sum().sum()  == 0:
    print("No Null Values")
else:
    print('Null Values Available')

print(f'  Columns in Dataset 2: {df2.columns.tolist()}')



No Null Values
  Columns in Dataset 2: ['Calories_Burned', 'Weight (kg)', 'Age', 'Gender', 'Session_Duration (hours)', 'Avg_BPM', 'BMI', 'Weather_Conditions', 'Exercise_Intensity']


# Setting Dataset Identifiers

In [51]:
df1['source'] = 0
df2['source'] = 1

print('Source flags added:')
print(f'  DS1 (source=0): {len(df1)} rows')
print(f'  DS2 (source=1): {len(df2)} rows')

Source flags added:
  DS1 (source=0): 973 rows
  DS2 (source=1): 3864 rows


# Removing Outliers Per Dataset

In [61]:
def remove_outliers_iqr(df, col, k=3.0):
    Q1, Q3 = df[col].quantile(0.25), df[col].quantile(0.75)
    IQR = Q3 - Q1
    lo, hi = Q1 - k*IQR, Q3 + k*IQR
    before = len(df)
    df_clean = df[(df[col] >= lo) & (df[col] <= hi)]
    print(f'  {col}: Removed {before - len(df_clean)} outliers '
          f'(IQR range [{lo:.0f}, {hi:.0f}])')
    return df_clean

print('Outlier removal DS1:')
df1 = remove_outliers_iqr(df1, 'Calories_Burned')
print('Outlier removal DS2:')
df2 = remove_outliers_iqr(df2, 'Calories_Burned')

Outlier removal DS1:
  Calories_Burned: Removed 0 outliers (IQR range [-348, 2144])
Outlier removal DS2:
  Calories_Burned: Removed 0 outliers (IQR range [-404, 1010])
